#Preparar datos sobre Naves Imperiales de Star Wars (v1)

In [ ]:
#@title Librerías

import sys, os, re, random
import numpy as np
import pandas as pd
from google.colab import files


import os
import csv

import ipywidgets as widgets
from ipywidgets import Box, Layout
from IPython.display import clear_output
import random

print("Librerías cargadas.")

Librerías cargadas.


#Cargar datos descargados

In [ ]:
# @title Acceder al Drive {"single-column":true}

# Nota: la primera vez se debe confirmar el uso logueandose en "Google Drive File Stream" y obteniendo código de autentificación.
from google.colab import drive
drive.mount('/content/gdrive', force_remount=True)

# directorio local en Google Drive
path = '/content/gdrive/MyDrive/demosColab/demoStarWars/datos/'  #@param {type:"string"}


Mounted at /content/gdrive


In [ ]:
#@title Cargar datos

#@markdown ### Archivo de datos a utilizar:
archivo_datos = 'navesOri.csv'  #@param {type:"string"}
#@markdown ### Configuración del archivo CSV:
delimitador_columnas = ',' #@param {type:"string"}
separador_decimal = '.' #@param {type:"string"}

# funciones auxiliares

# importa datos
def cargarDatosDF(path, archivo_datos, delimitador_columnas, separador_decimal, mostrarEstadisticas=True):
  if os.path.isfile( path + '/' + archivo_datos ):
    # existe el archivo
    if ((delimitador_columnas is None) or (delimitador_columnas=="")):
      # si no se define asume ","
      delimitador_columnas = ","
    if ((separador_decimal is None) or (separador_decimal=="")):
      # si no se define asume "."
      separador_decimal = "."
    if (delimitador_columnas==separador_decimal):
      # ambos delimitadores iguales, cambia el decimal
      if delimitador_columnas == ",":
        separador_decimal = "."
      else:
        separador_decimal = ","
      print("- Ambos delimitadores configurados igual, se cambia separador decimal a '" + separador_decimal + "'!")
    # carga datos
    df = pd.read_csv(path + archivo_datos,
                     sep=delimitador_columnas, decimal=separador_decimal,
                     skip_blank_lines=True,
                     engine="python")
    print("> Archivo de datos", archivo_datos, "cargado")
    # muestra estadísticas
    if mostrarEstadisticas:
      print("\n> Cabecera: ")
      print(df.head())
      print("\n> Características: ")
      print(df.describe())
      print("\n")
    # controla que el archivo tenga sentido
    if len(df.columns.values.tolist())<2:
      print("\n> El archivo de datos debería tener al menos 2 columnas, revise delimitador de columnas!")
      return None
    else:
      return df
  else:
    print("No existe archivo de datos ", archivo_datos, "!")
    return None

# importa definición axiliar de las clases (si existe)
def cargarNombreClases(path, archivo_datos):
  # importa definición de la clase
  arClasesFN = archivo_datos.split('.')[0] + '_nombreClases.txt'
  if os.path.isfile( path + '/' + arClasesFN ):
    # si existe, carga los datos
    with open( path + '/' + arClasesFN, mode='r') as csvfile:
        r = csv.reader(csvfile, delimiter=',')
        auxAtributo = r.__next__()
        auxClases = r.__next__()
    print('\n> Definición de los valores discretos para la clase cargada de ' + arClasesFN +'.\n')
    return auxAtributo[0], ','.join(auxClases)
  else:
    # no encontrado
    return "", ""


# configura para que muestre todas las columnas y filas
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

## aplicación de los parámetros elegidos

# Carga los datos del CSV y muestra los primeros
df = cargarDatosDF(path, archivo_datos,
                   delimitador_columnas, separador_decimal, mostrarEstadisticas=True)



> Archivo de datos navesOri.csv cargado

> Cabecera: 
                         Name           Ship Type  \
0             Builder Shuttle       Landing Craft   
1                      TIE X3        TIE Fighters   
2  Star Galleon Class Frigate        Medium Ships   
3               A-9 Vigilance  Other Starfighters   
4        TIE Ground Targeting         TIE Bombers   

                               Model          Manufacturer      Length Crew  \
0  Builder Shuttle Mark 1 and Mark 2     Cygnus Spaceworks   40 meters    4   
1                     TIE/x3 Fighter  Sienar Fleet Systems  7.8 meters    1   
2         Star Galleon Class Frigate      Kuat Drive Yards  298 meters  150   
3                      A-9 Vigilance      Kuat Drive Yards  7.4 meters    1   
4                     TIE/gt Fighter  Sienar Fleet Systems  6.3 meters    1   

                                     Cargo Capacity Consumables  \
0  15 metric tons (Mark 2 can hold 400 metric tons)      1 week   
1                 

In [ ]:
#@title Mostrar Estadísticas de datos recolectados


# variables auxiliares
atributo_clase = ""

# configura para que muestre todas las columnas y filas
pd.options.display.max_rows = 100
pd.options.display.max_columns = 100

# devuelve listas de columnas numéricas y no numéricas
def devolNombreColumnas(ndf):
  colValues = []
  colNoValues = []
  for col in ndf.columns:
    if ndf[col].dtypes in ("object", "bool"):
      colNoValues.append( col )
    else:
      colValues.append( col )
  return colValues, colNoValues

# función auxiliar para separar datos de entrada y de salida
def separarDatosXY(ndf, atributo_clase="", xSoloNros=True):
  # hace una copia auxiliar del data frame
  cdf = ndf.copy()
  # saca el atributo clase (OPCIONAL)
  if atributo_clase == "":
    Y = []
  else:
    # datos atributo clase
    Y = np.array( cdf.pop(atributo_clase).fillna("-NAN-") )
  if xSoloNros:
    # se queda sólo con los atributos numéricos (OPCIONAL)
    for col in cdf.columns:
      if cdf[col].dtypes == "object":
          cdf.pop( col )
  # datos de entrada
  X = np.array(cdf.infer_objects(copy=False).fillna(0.001))
  return X, Y, np.array(cdf.columns)

def convColsNumericas(ndf, atributos_no_convertir = []):
  # hace una copia auxiliar del data frame
  cdf = ndf.copy()
  # convierte todas las no numéricas a numéricas (OPCIONAL)
  for col in cdf.columns:
    if col not in atributos_no_convertir:
      if cdf[col].dtypes == "object":
        # genera diccionario de valores
        valores = cdf[col].unique()
        diccValores = dict(zip(valores, range(len(valores))))
        # realiza el reemplazo
        cdf[col] = cdf[col].map(lambda s: diccValores.get(s) if s in diccValores else s)
  return cdf

# función auxiliar
def generar_estadisticas_detalladas(orDF, titulo=""):
  # título
  print("\n", titulo, ": ")
  # obtiene las estadísticas generales
  estDF = orDF.describe().transpose()
  #  genera y formatea las estadísticas
  if "min" in estDF and "max" in estDF:
    rangoValores = "[ " + estDF["min"].apply('{:.2f}'.format) + " ; " + estDF["max"].apply('{:.2f}'.format) + " ]"
  else:
    rangoValores = estDF["unique"].infer_objects(copy=False).fillna(0.0).apply('{:.0f}'.format)
  rangoValores.name = "Rango Valores"
  # para campos no numéricos muestra las cantidades por valor
  for col in orDF.columns:
    if orDF[col].dtypes in ("object", "bool"):
      auxStr = str( orDF[col].value_counts() ).replace("\n", " ; ")
      if (auxStr.index("Name")-3) > 0:
        # saca lo del final porque no sirve
        auxStr = auxStr[:auxStr.index("Name")-3]
      rangoValores[col] = "{ " + auxStr + " }"
  if "mean" in estDF and "std" in estDF:
    promValores = estDF["mean"].fillna(0.0).apply('{:.3f}'.format) + " ± " + estDF["std"].fillna(0.0).apply('{:.3f}'.format)
  else:
    promValores = estDF["count"].apply('{:.0f}'.format)
  promValores.name = "Promedio ± Desvío"
  # obtiene valores "ceros" y nulos
  zero_val = (orDF == 0.00).astype(int).sum(axis=0)
  zero_val.name = "¿Valores Ceros?"
  mis_val = orDF.isnull().sum()
  mis_val.name = "¿Valores Nulos?"
  # prepara la nueva tabla para mostrar
  nTable = pd.concat([orDF.dtypes, rangoValores, promValores, zero_val, mis_val], axis=1)
  nTable = nTable.rename( columns = {0: 'Tipo Valor',  1: 'Rango Valores', 2: 'Promedio ± Desvío', 3: '¿Valores Ceros?', 4: '¿Valores Nulos?' } )
  # muestra la nueva tabla
  pd.set_option('max_colwidth', None)
  display(nTable.fillna("-"))
  print("Tiene " + str(orDF.shape[1]) + " atributos y " + str(orDF.shape[0]) + " ejemplos.")
  print("\n")
  return

# muestra las estadísticas
generar_estadisticas_detalladas(df, "> Estadísticas de los datos recolectados")



 > Estadísticas de los datos recolectados : 


Tipo Valor  \
Name                      object   
Ship Type                 object   
Model                     object   
Manufacturer              object   
Length                    object   
Crew                      object   
Cargo Capacity            object   
Consumables               object   
Hyperdrive Multiplier     object   
Hyperdrive Backup         object   
Speed                     object   
Hull                      object   
Special Features          object   
URL_data                  object   
URL_image                 object   
Shields                   object   
Weapons                   object   
Troops                    object   
Onboard Craft             object   

                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                     

Tiene 19 atributos y 107 ejemplos.




#Preparar Datos de Naves Imperiales

## Ajustar Formato de Campos

In [ ]:
#@title Preparar Campos donde se extrae parte numérica eliminando texto

# Hace copia de trabajo (para romper datos extraidos)
ndf = df.copy()

# funciones auxiliares
def extraerPrimerNro(columnName):
  global ndf
  # siempre elimina "," como separador de miles
  reemplazar(columnName, ",")
  # toma número con decimales
  ndf[columnName] = ndf[columnName].str.extract(r'(\d+\.?\d*)\D*', expand=True)
  return

def reemplazar(columnName, oldValue, newValue=""):
  global ndf
  ndf[columnName] = ndf[columnName].str.replace(oldValue, newValue, regex=False)
  ndf[columnName] = ndf[columnName].fillna(0)
  return

def completarVacios(columnName, newValue=-1):
  global ndf
  ndf[columnName] = ndf[columnName].fillna(newValue)
  return

def convertirFloat(columnName):
  global ndf
  ndf[columnName] = ndf[columnName].astype(float)

print("-procesa Length")
reemplazar("Length", " meters")
extraerPrimerNro("Length")
completarVacios("Length", 0)
convertirFloat("Length")

print("-procesa Crew")
extraerPrimerNro("Crew")
completarVacios("Crew", 0)
convertirFloat("Crew")

print("-procesa Troops")
extraerPrimerNro("Troops")
completarVacios("Troops", 0)
convertirFloat("Troops")

print("-procesa Hyperdrive Multiplier")
extraerPrimerNro("Hyperdrive Multiplier")
completarVacios("Hyperdrive Multiplier", -1)
convertirFloat("Hyperdrive Multiplier")

print("-procesa Hyperdrive Backup")
extraerPrimerNro("Hyperdrive Backup")
completarVacios("Hyperdrive Backup", -1)
convertirFloat("Hyperdrive Backup")

print("-procesa Speed")
reemplazar("Speed", " MGLT")
extraerPrimerNro("Speed")
completarVacios("Speed", 0)
convertirFloat("Speed")

print("-procesa Hull")
reemplazar("Hull", " RU")
extraerPrimerNro("Hull")
completarVacios("Hull", 0)
convertirFloat("Hull")

print("-procesa Shields")
reemplazar("Shields", " SBD")
extraerPrimerNro("Shields")
completarVacios("Shields", -1)
convertirFloat("Shields")

# muestra cambios
ndf.head()

-procesa Length
-procesa Crew
-procesa Troops
-procesa Hyperdrive Multiplier
-procesa Hyperdrive Backup
-procesa Speed
-procesa Hull
-procesa Shields


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,Cygnus Spaceworks,40.0,4.0,15 metric tons (Mark 2 can hold 400 metric tons),1 week,2.0,2.0,35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,0.0,NaN
1,TIE X3,TIE Fighters,TIE/x3 Fighter,Sienar Fleet Systems,7.8,1.0,150 kilograms,5 days,-1.0,-1.0,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,2 Laser Cannons,0.0,NaN
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,Kuat Drive Yards,298.0,150.0,"100,000 metric tons",6 months,2.0,15.0,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,10 Turbolaser Cannons and 1 Concussion Missile Launcher.,300.0,NaN
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,Kuat Drive Yards,7.4,1.0,55 kilograms,1 day,-1.0,-1.0,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,2 Laser Cannons,0.0,NaN
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,Sienar Fleet Systems,6.3,1.0,65 kilograms,2 days,-1.0,-1.0,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,"1 Laser Cannon, 1 Concussion Missile Launcher and 1 General Purpose Launcher.",0.0,NaN


In [ ]:
#@title Preparar campo Cargo Capacity

print("-procesa Cargo Capacity (pasa todo a kilogramos)")

# realiza los cambios considerando tipo de métrica
auxOriList = list(df["Cargo Capacity"])
auxNewList = []
for val in auxOriList:
  #print(val)
  if (val is None) or (str(val) in ["nan", "(varies according to mission profile)"]):
    auxNewList.append( -1 )
  elif str(val)=="Mu-1 and Mu-2: 100 metric tons / Mu-3: 50 metric tons":
    auxNewList.append( 75 )
  else:
    val = val.replace(",", "")
    val = val.replace(" + 1", "")
    if "(" in val:
      posParent = val.index("(")
    else:
      posParent = -1
    if "-" in val:
      # calcula promedio
      posSep1 = val.index("-")
      posSep2 = val.index(" ")
      nval = ( float(val[0:posSep1]) + float(val[posSep1+1:posSep2]) ) / 2
      val = str(nval) + val[posSep2:]
    if "or" in val:
      # calcula promedio
      posSep1 = val.index(" or ")
      posSep2 = val[posSep1+4:].index(" ") + posSep1 + 4
      nval = ( float(val[0:posSep1]) + float(val[posSep1+4:posSep2]) ) / 2
      val = str(nval) + val[posSep2:]
    if "kilograms" in val:
      posAux = val.index("kilograms")
      if (posParent>0) and (posParent<posAux):
        posAux = posParent
      auxNewList.append( float(val[:posAux].strip()) )
    elif "metric ton" in val:
      posAux = val.index("metric ton")
      if (posParent>0) and (posParent<posAux):
        posAux = posParent
      auxNewList.append( float(val[:posAux].strip()) * 1000 )
    else:
      print("No se puede procesar: ", val)

# actualiza los datos
ndf["Cargo Capacity"] = auxNewList
convertirFloat("Cargo Capacity")


ndf.head()

-procesa Cargo Capacity (pasa todo a kilogramos)


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,Cygnus Spaceworks,40.0,4.0,15000.0,1 week,2.0,2.0,35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,0.0,NaN
1,TIE X3,TIE Fighters,TIE/x3 Fighter,Sienar Fleet Systems,7.8,1.0,150.0,5 days,-1.0,-1.0,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,2 Laser Cannons,0.0,NaN
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,Kuat Drive Yards,298.0,150.0,100000000.0,6 months,2.0,15.0,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,10 Turbolaser Cannons and 1 Concussion Missile Launcher.,300.0,NaN
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,Kuat Drive Yards,7.4,1.0,55.0,1 day,-1.0,-1.0,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,2 Laser Cannons,0.0,NaN
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,Sienar Fleet Systems,6.3,1.0,65.0,2 days,-1.0,-1.0,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,"1 Laser Cannon, 1 Concussion Missile Launcher and 1 General Purpose Launcher.",0.0,NaN


In [ ]:
#@title Preparar campo Consumables

print("-procesa Consumables (pasa todo a meses)")

# realiza los cambios considerando tipo de métrica
auxOriList = list(df["Consumables"])
auxNewList = []
for val in auxOriList:
  #print(val)
  if (val is None) or (str(val) in ["nan", "(varies according to mission profile)"]):
    auxNewList.append( -1 )
  if str(val) == "Mu-1 and Mu-2: 6 months / Mu-3: 2 months":
    auxNewList.append( 4 )
  else:
    val = val.replace(",", "")
    #val = val.replace(" + 1", "")
    if "(" in val:
      posParent = val.index("(")
    else:
      posParent = -1
    if "-" in val:
      # calcula promedio
      posSep1 = val.index("-")
      posSep2 = val.index(" ")
      nval = ( float(val[0:posSep1]) + float(val[posSep1+1:posSep2]) ) / 2
      val = str(nval) + val[posSep2:]
    if "or" in val:
      # calcula promedio
      posSep1 = val.index(" or ")
      posSep2 = val[posSep1+4:].index(" ") + posSep1 + 4
      nval = ( float(val[0:posSep1]) + float(val[posSep1+4:posSep2]) ) / 2
      val = str(nval) + val[posSep2:]
    if "month" in val:
      posAux = val.index("month")
      if (posParent>0) and (posParent<posAux):
        posAux = posParent
      auxNewList.append( float(val[:posAux].strip()) )
    elif "year" in val:
      posAux = val.index("year")
      if (posParent>0) and (posParent<posAux):
        posAux = posParent
      auxNewList.append( float(val[:posAux].strip()) * 12 )
    elif "week" in val:
      posAux = val.index("week")
      if (posParent>0) and (posParent<posAux):
        posAux = posParent
      auxNewList.append( float(val[:posAux].strip()) / 4.5 )
    elif "day" in val:
      posAux = val.index("day")
      if (posParent>0) and (posParent<posAux):
        posAux = posParent
      auxNewList.append( float(val[:posAux].strip()) / 30 )
    else:
      print("No se puede procesar: ", val)

# actualiza los datos
ndf["Consumables"] = auxNewList
convertirFloat("Consumables")

ndf.head()

-procesa Consumables (pasa todo a meses)


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,Cygnus Spaceworks,40.0,4.0,15000.0,0.222222,2.0,2.0,35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,0.0,NaN
1,TIE X3,TIE Fighters,TIE/x3 Fighter,Sienar Fleet Systems,7.8,1.0,150.0,0.166667,-1.0,-1.0,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,2 Laser Cannons,0.0,NaN
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,Kuat Drive Yards,298.0,150.0,100000000.0,6.000000,2.0,15.0,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,10 Turbolaser Cannons and 1 Concussion Missile Launcher.,300.0,NaN
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,Kuat Drive Yards,7.4,1.0,55.0,0.033333,-1.0,-1.0,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,2 Laser Cannons,0.0,NaN
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,Sienar Fleet Systems,6.3,1.0,65.0,0.066667,-1.0,-1.0,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,"1 Laser Cannon, 1 Concussion Missile Launcher and 1 General Purpose Launcher.",0.0,NaN


In [ ]:
#@title Preparar campo Manufacturer

print("-procesa Manufacturer (asigna codigo valor ID)")

from sklearn.preprocessing import LabelEncoder

encoder = LabelEncoder().fit(df["Manufacturer"])
ndf["Manufacturer"] = encoder.transform(df["Manufacturer"])
convertirFloat("Manufacturer")

# muestra códificación asignada
print("\n\tCodificación asignada: ")
for i in range(len(encoder.classes_)):
  print("\t\t", i, ":", encoder.classes_[i])
print("")

ndf.head()

-procesa Manufacturer (asigna codigo valor ID)

	Codificación asignada: 
		 0 : Byss Worx / Imperial Department of Military Research
		 1 : CPG Space Products
		 2 : Corellian Engineering Corporation
		 3 : Cygnus Spaceworks
		 4 : Cygnus Spaceworks / Sienar Fleet Systems
		 5 : Damorian Manufacturing Corporation
		 6 : Imperial Shipyards
		 7 : Incom Corporation
		 8 : Kuat Drive Yards
		 9 : Loronar
		 10 : Loronar / Rendili StarDrive /Sienar Fleet Systems and Kuat Drive Yards
		 11 : Meller & Dax
		 12 : Mesens Corporation
		 13 : Rendili StarDrive
		 14 : Republic Sienar Systems
		 15 : Rothana Heavy Engineering
		 16 : Rothana Heavy Engineering / Kuat Drive Yards
		 17 : Santhe / Sienar Technologies
		 18 : Sienar Fleet Systems
		 19 : Sienar Fleet Systems / Chiss Ascendancy
		 20 : Sienar Fleet Systems / Imperial Department of Military Research
		 21 : Sienar Fleet Systems / Shobquix Yards
		 22 : Sienar Fleet Systems / Zsinj Development Incorporated
		 23 : Sienar Fleet Systems/

,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Weapons,Troops,Onboard Craft
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,3.0,40.0,4.0,15000.0,0.222222,2.0,2.0,35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,NaN,0.0,NaN
1,TIE X3,TIE Fighters,TIE/x3 Fighter,18.0,7.8,1.0,150.0,0.166667,-1.0,-1.0,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,2 Laser Cannons,0.0,NaN
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,8.0,298.0,150.0,100000000.0,6.000000,2.0,15.0,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,10 Turbolaser Cannons and 1 Concussion Missile Launcher.,300.0,NaN
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,8.0,7.4,1.0,55.0,0.033333,-1.0,-1.0,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,2 Laser Cannons,0.0,NaN
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,18.0,6.3,1.0,65.0,0.066667,-1.0,-1.0,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,"1 Laser Cannon, 1 Concussion Missile Launcher and 1 General Purpose Launcher.",0.0,NaN


In [ ]:
#@title Preparar campo Weapons

print("-procesa Weapons (extrae lista de armas y genera nuevas columnas numéricas)")

import re

newValues = []

for f in df["Weapons"]:
  # extrae la lista de armas
  auxSplit =  re.split(", | and ", str(f))
  #print(auxSplit)
  # define columnas en base subcadenas obtenidas
  auxDict = {}
  for w in auxSplit:
    if w=="nan":
      continue
    res = re.search('(\d+\.?\d*)\s(.+)', w)
    if (res is not None):
      ##print(res.group(1), "*", res.group(2))
      auxW = str(res.group(2)).replace(".", "").upper().strip()
      if "(" in auxW:
        # saca toda aclaración entre paréntesis
        auxW = auxW[:auxW.find("(")].strip()
      # si no es plural, le agrega S (para unificar columnas)
      if auxW[len(auxW)-1] != "S":
        auxW = auxW + "S"
      auxQ = float(res.group(1))
    else:
      auxW = w.upper().strip()
      auxQ = 1.0
    if len(auxW)<1:
      auxW = "OTHER"
    if auxW in auxDict:
      auxDict[auxW] = auxDict[auxW] + auxQ
    else:
      auxDict[auxW] = auxQ

  newValues.append( auxDict )


##print(newValues)

# crea nuevo data frame
auxDF = pd.DataFrame.from_dict(newValues)
# completa valores vacios con cero
for col in auxDF:
  auxDF[col] = auxDF[col].fillna(0)
# agrega al data frame anterior y elimina columna original
ndf = pd.concat([ndf, auxDF], axis=1)
ndf = ndf.drop(["Weapons"], axis=1)

ndf.head()

-procesa Weapons (extrae lista de armas y genera nuevas columnas numéricas)


<>:18: SyntaxWarning: invalid escape sequence '\d'
<>:18: SyntaxWarning: invalid escape sequence '\d'
/tmp/ipykernel_707/3071266889.py:18: SyntaxWarning: invalid escape sequence '\d'
  res = re.search('(\d+\.?\d*)\s(.+)', w)


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Troops,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,CONCUSSION MISSILE LAUNCHERS,GENERAL PURPOSE LAUNCHERS,HEAVY TURBOLASER TURRETS,DOUBLE LASER CANNONS,DOUBLE BLASTER CANNONS,LASER CANNON TURRETS,TRACTOR BEAM PROJECTORS,TWIN LASER CANNONS,QUAD TURBOLASER BATTERIES,DOUBLE TURBOLASER BATTERIES,CUNCUSSION MISSILE LAUNCHERS,ION CANNONS,MU-1,LASER CANNONS / MU-3: 2 MEDIUM LASER CANNONS,LIGHT LASER CANNONS,DOUBLE TURBOLASER TURRETS,MEDIUM ION CANNONS,PROTON TORPEDO LAUNCHERS,HEAVY TURBOLASER BATTERIES,HEAVY LASER CANNONS,GRAVITY WELL PROJECTORS,SUPERLASERS,QUAD TURBOLASER CANNONS,TURBOLASER BATTERIES,MEDIUM LASER CANNONS,TWIN BLASTER CANNON TURRETS,PROTON TORPEDOS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHER.,BLASTER CANNONS,DOUBLE TURBOLASER CANNONS,MEDIUM TURBOLASERS,HEAVY BLASTER CANNONS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHERS.,ION CANNON TURRETS,REPEATING BLASTER TURRETS,BUILD-IN SPACE BOMB.,HEAVY LASER CANNON TURRETS,HEAVY TURBOLASER CANNONS,TURBOLASER CANNON TURRETS,THERMAL DETONATOR LAUNCHERS,DOUBLE BLASTER CANNON TURRETS,LASER CANNONS TURRETS,QUAD LASER CANNONS,ION CANNON BATTERIES,HEAVY TRACTORBEAM PROJECTORS,HULL-CUTTING AIRLOCKS,QUAD BLASTER CANNON,EMP CANNONS,MEDIUM BLASTER CANNONS,LIGHT TURBOLASER CANNONS,(VARIES ACCORDING TO SHIP MODEL),DOUBLE HEAVY LASER TURRETS,QUAD LASER CANNON TURRETS,HEAVY TURBOLASERS,MEDIUM DUAL TURBOLASERS,PROTON TORPEDO TUBES,DOUBLE LASER CANNON TURRETS,QUAD TURBOLASER TURRETS,TURBOLASER TURRETS
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,3.0,40.0,4.0,15000.0,0.222222,2.0,2.0,35.0,45.0,4 Adjustable electromagnetic lifter arms.,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TIE X3,TIE Fighters,TIE/x3 Fighter,18.0,7.8,1.0,150.0,0.166667,-1.0,-1.0,110.0,14.0,NaN,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,0.0,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,8.0,298.0,150.0,100000000.0,6.000000,2.0,15.0,18.0,228.0,Detachable cargo hold with hyperspace capability.,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,300.0,NaN,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,8.0,7.4,1.0,55.0,0.033333,-1.0,-1.0,115.0,16.0,NaN,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,0.0,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,18.0,6.3,1.0,65.0,0.066667,-1.0,-1.0,75.0,9.0,NaN,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,0.0,NaN,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,

In [ ]:
#@title Preparar campo Special Features

# función auxiliar para clusterizar datos
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score

def clusterizarDatosString(colunmnName, cantClusters=5):
  global ndf
  ndf[colunmnName] = ndf[colunmnName].fillna("-")
  documents  = list( ndf[colunmnName] )

  vectorizer = TfidfVectorizer(stop_words='english')
  X = vectorizer.fit_transform(documents)

  model = KMeans(n_clusters=cantClusters, init='k-means++', max_iter=100, n_init=1)
  ndf[colunmnName] = model.fit_transform(X)



print("-procesa Special Features (usando clustering)")

clusterizarDatosString("Special Features")
convertirFloat("Special Features")

ndf.head()

-procesa Special Features (usando clustering)


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Troops,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,CONCUSSION MISSILE LAUNCHERS,GENERAL PURPOSE LAUNCHERS,HEAVY TURBOLASER TURRETS,DOUBLE LASER CANNONS,DOUBLE BLASTER CANNONS,LASER CANNON TURRETS,TRACTOR BEAM PROJECTORS,TWIN LASER CANNONS,QUAD TURBOLASER BATTERIES,DOUBLE TURBOLASER BATTERIES,CUNCUSSION MISSILE LAUNCHERS,ION CANNONS,MU-1,LASER CANNONS / MU-3: 2 MEDIUM LASER CANNONS,LIGHT LASER CANNONS,DOUBLE TURBOLASER TURRETS,MEDIUM ION CANNONS,PROTON TORPEDO LAUNCHERS,HEAVY TURBOLASER BATTERIES,HEAVY LASER CANNONS,GRAVITY WELL PROJECTORS,SUPERLASERS,QUAD TURBOLASER CANNONS,TURBOLASER BATTERIES,MEDIUM LASER CANNONS,TWIN BLASTER CANNON TURRETS,PROTON TORPEDOS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHER.,BLASTER CANNONS,DOUBLE TURBOLASER CANNONS,MEDIUM TURBOLASERS,HEAVY BLASTER CANNONS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHERS.,ION CANNON TURRETS,REPEATING BLASTER TURRETS,BUILD-IN SPACE BOMB.,HEAVY LASER CANNON TURRETS,HEAVY TURBOLASER CANNONS,TURBOLASER CANNON TURRETS,THERMAL DETONATOR LAUNCHERS,DOUBLE BLASTER CANNON TURRETS,LASER CANNONS TURRETS,QUAD LASER CANNONS,ION CANNON BATTERIES,HEAVY TRACTORBEAM PROJECTORS,HULL-CUTTING AIRLOCKS,QUAD BLASTER CANNON,EMP CANNONS,MEDIUM BLASTER CANNONS,LIGHT TURBOLASER CANNONS,(VARIES ACCORDING TO SHIP MODEL),DOUBLE HEAVY LASER TURRETS,QUAD LASER CANNON TURRETS,HEAVY TURBOLASERS,MEDIUM DUAL TURBOLASERS,PROTON TORPEDO TUBES,DOUBLE LASER CANNON TURRETS,QUAD TURBOLASER TURRETS,TURBOLASER TURRETS
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,3.0,40.0,4.0,15000.0,0.222222,2.0,2.0,35.0,45.0,0.998658,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,0.0,NaN,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TIE X3,TIE Fighters,TIE/x3 Fighter,18.0,7.8,1.0,150.0,0.166667,-1.0,-1.0,110.0,14.0,0.037238,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,0.0,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,8.0,298.0,150.0,100000000.0,6.000000,2.0,15.0,18.0,228.0,0.987765,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,300.0,NaN,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,8.0,7.4,1.0,55.0,0.033333,-1.0,-1.0,115.0,16.0,0.037238,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,0.0,NaN,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,18.0,6.3,1.0,65.0,0.066667,-1.0,-1.0,75.0,9.0,0.037238,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,0.0,NaN,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#@title Preparar campo Onboard Craft

print("-procesa Onboard Craft (usando clustering)")

clusterizarDatosString("Onboard Craft")
convertirFloat("Onboard Craft")

ndf.head()

-procesa Onboard Craft (usando clustering)


,Name,Ship Type,Model,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,URL_data,URL_image,Shields,Troops,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,CONCUSSION MISSILE LAUNCHERS,GENERAL PURPOSE LAUNCHERS,HEAVY TURBOLASER TURRETS,DOUBLE LASER CANNONS,DOUBLE BLASTER CANNONS,LASER CANNON TURRETS,TRACTOR BEAM PROJECTORS,TWIN LASER CANNONS,QUAD TURBOLASER BATTERIES,DOUBLE TURBOLASER BATTERIES,CUNCUSSION MISSILE LAUNCHERS,ION CANNONS,MU-1,LASER CANNONS / MU-3: 2 MEDIUM LASER CANNONS,LIGHT LASER CANNONS,DOUBLE TURBOLASER TURRETS,MEDIUM ION CANNONS,PROTON TORPEDO LAUNCHERS,HEAVY TURBOLASER BATTERIES,HEAVY LASER CANNONS,GRAVITY WELL PROJECTORS,SUPERLASERS,QUAD TURBOLASER CANNONS,TURBOLASER BATTERIES,MEDIUM LASER CANNONS,TWIN BLASTER CANNON TURRETS,PROTON TORPEDOS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHER.,BLASTER CANNONS,DOUBLE TURBOLASER CANNONS,MEDIUM TURBOLASERS,HEAVY BLASTER CANNONS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHERS.,ION CANNON TURRETS,REPEATING BLASTER TURRETS,BUILD-IN SPACE BOMB.,HEAVY LASER CANNON TURRETS,HEAVY TURBOLASER CANNONS,TURBOLASER CANNON TURRETS,THERMAL DETONATOR LAUNCHERS,DOUBLE BLASTER CANNON TURRETS,LASER CANNONS TURRETS,QUAD LASER CANNONS,ION CANNON BATTERIES,HEAVY TRACTORBEAM PROJECTORS,HULL-CUTTING AIRLOCKS,QUAD BLASTER CANNON,EMP CANNONS,MEDIUM BLASTER CANNONS,LIGHT TURBOLASER CANNONS,(VARIES ACCORDING TO SHIP MODEL),DOUBLE HEAVY LASER TURRETS,QUAD LASER CANNON TURRETS,HEAVY TURBOLASERS,MEDIUM DUAL TURBOLASERS,PROTON TORPEDO TUBES,DOUBLE LASER CANNON TURRETS,QUAD TURBOLASER TURRETS,TURBOLASER TURRETS
0,Builder Shuttle,Landing Craft,Builder Shuttle Mark 1 and Mark 2,3.0,40.0,4.0,15000.0,0.222222,2.0,2.0,35.0,45.0,0.998658,http://insd.swcombine.com/insd/builder.htm,http://insd.swcombine.com/insd/Ship_files/BUILDER.JPG,-1.0,0.0,0.030741,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TIE X3,TIE Fighters,TIE/x3 Fighter,18.0,7.8,1.0,150.0,0.166667,-1.0,-1.0,110.0,14.0,0.037238,http://insd.swcombine.com/insd/tiex3.htm,http://insd.swcombine.com/insd/Ship_files/TIEADVA3.JPG,24.0,0.0,0.030741,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Star Galleon Class Frigate,Medium Ships,Star Galleon Class Frigate,8.0,298.0,150.0,100000000.0,6.000000,2.0,15.0,18.0,228.0,0.987765,http://insd.swcombine.com/insd/stargal.htm,http://insd.swcombine.com/insd/Ship_files/STARGAL.JPG,320.0,300.0,0.030741,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,A-9 Vigilance,Other Starfighters,A-9 Vigilance,8.0,7.4,1.0,55.0,0.033333,-1.0,-1.0,115.0,16.0,0.037238,http://insd.swcombine.com/insd/a9vigil.htm,http://insd.swcombine.com/insd/Ship_files/TIERPT.JPG,-1.0,0.0,0.030741,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TIE Ground Targeting,TIE Bombers,TIE/gt Fighter,18.0,6.3,1.0,65.0,0.066667,-1.0,-1.0,75.0,9.0,0.037238,http://insd.swcombine.com/insd/tiegt.htm,http://insd.swcombine.com/insd/Ship_files/TIEGT.JPG,-1.0,0.0,0.030741,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


## Organizar Campos

In [ ]:
#@title Eliminar campos Name, Model y URLs

ndf = ndf.drop(columns=["Name", "Model", "URL_data", "URL_image"])

ndf.head()

,Ship Type,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,Shields,Troops,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,CONCUSSION MISSILE LAUNCHERS,GENERAL PURPOSE LAUNCHERS,HEAVY TURBOLASER TURRETS,DOUBLE LASER CANNONS,DOUBLE BLASTER CANNONS,LASER CANNON TURRETS,TRACTOR BEAM PROJECTORS,TWIN LASER CANNONS,QUAD TURBOLASER BATTERIES,DOUBLE TURBOLASER BATTERIES,CUNCUSSION MISSILE LAUNCHERS,ION CANNONS,MU-1,LASER CANNONS / MU-3: 2 MEDIUM LASER CANNONS,LIGHT LASER CANNONS,DOUBLE TURBOLASER TURRETS,MEDIUM ION CANNONS,PROTON TORPEDO LAUNCHERS,HEAVY TURBOLASER BATTERIES,HEAVY LASER CANNONS,GRAVITY WELL PROJECTORS,SUPERLASERS,QUAD TURBOLASER CANNONS,TURBOLASER BATTERIES,MEDIUM LASER CANNONS,TWIN BLASTER CANNON TURRETS,PROTON TORPEDOS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHER.,BLASTER CANNONS,DOUBLE TURBOLASER CANNONS,MEDIUM TURBOLASERS,HEAVY BLASTER CANNONS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHERS.,ION CANNON TURRETS,REPEATING BLASTER TURRETS,BUILD-IN SPACE BOMB.,HEAVY LASER CANNON TURRETS,HEAVY TURBOLASER CANNONS,TURBOLASER CANNON TURRETS,THERMAL DETONATOR LAUNCHERS,DOUBLE BLASTER CANNON TURRETS,LASER CANNONS TURRETS,QUAD LASER CANNONS,ION CANNON BATTERIES,HEAVY TRACTORBEAM PROJECTORS,HULL-CUTTING AIRLOCKS,QUAD BLASTER CANNON,EMP CANNONS,MEDIUM BLASTER CANNONS,LIGHT TURBOLASER CANNONS,(VARIES ACCORDING TO SHIP MODEL),DOUBLE HEAVY LASER TURRETS,QUAD LASER CANNON TURRETS,HEAVY TURBOLASERS,MEDIUM DUAL TURBOLASERS,PROTON TORPEDO TUBES,DOUBLE LASER CANNON TURRETS,QUAD TURBOLASER TURRETS,TURBOLASER TURRETS
0,Landing Craft,3.0,40.0,4.0,15000.0,0.222222,2.0,2.0,35.0,45.0,0.998658,-1.0,0.0,0.030741,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
1,TIE Fighters,18.0,7.8,1.0,150.0,0.166667,-1.0,-1.0,110.0,14.0,0.037238,24.0,0.0,0.030741,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
2,Medium Ships,8.0,298.0,150.0,100000000.0,6.000000,2.0,15.0,18.0,228.0,0.987765,320.0,300.0,0.030741,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
3,Other Starfighters,8.0,7.4,1.0,55.0,0.033333,-1.0,-1.0,115.0,16.0,0.037238,-1.0,0.0,0.030741,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0
4,TIE Bombers,18.0,6.3,1.0,65.0,0.066667,-1.0,-1.0,75.0,9.0,0.037238,-1.0,0.0,0.030741,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0


In [ ]:
#@title Reordena columnas para que Ship Type quede al final

ndf = ndf.reindex(columns=(list([a for a in ndf.columns if a != 'Ship Type'] + ['Ship Type']) ))

ndf.head()

,Manufacturer,Length,Crew,Cargo Capacity,Consumables,Hyperdrive Multiplier,Hyperdrive Backup,Speed,Hull,Special Features,Shields,Troops,Onboard Craft,LASER CANNONS,TURBOLASER CANNONS,CONCUSSION MISSILE LAUNCHERS,GENERAL PURPOSE LAUNCHERS,HEAVY TURBOLASER TURRETS,DOUBLE LASER CANNONS,DOUBLE BLASTER CANNONS,LASER CANNON TURRETS,TRACTOR BEAM PROJECTORS,TWIN LASER CANNONS,QUAD TURBOLASER BATTERIES,DOUBLE TURBOLASER BATTERIES,CUNCUSSION MISSILE LAUNCHERS,ION CANNONS,MU-1,LASER CANNONS / MU-3: 2 MEDIUM LASER CANNONS,LIGHT LASER CANNONS,DOUBLE TURBOLASER TURRETS,MEDIUM ION CANNONS,PROTON TORPEDO LAUNCHERS,HEAVY TURBOLASER BATTERIES,HEAVY LASER CANNONS,GRAVITY WELL PROJECTORS,SUPERLASERS,QUAD TURBOLASER CANNONS,TURBOLASER BATTERIES,MEDIUM LASER CANNONS,TWIN BLASTER CANNON TURRETS,PROTON TORPEDOS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHER.,BLASTER CANNONS,DOUBLE TURBOLASER CANNONS,MEDIUM TURBOLASERS,HEAVY BLASTER CANNONS,ORBITAL MINE OR THERMAL DETONATOR LAUNCHERS.,ION CANNON TURRETS,REPEATING BLASTER TURRETS,BUILD-IN SPACE BOMB.,HEAVY LASER CANNON TURRETS,HEAVY TURBOLASER CANNONS,TURBOLASER CANNON TURRETS,THERMAL DETONATOR LAUNCHERS,DOUBLE BLASTER CANNON TURRETS,LASER CANNONS TURRETS,QUAD LASER CANNONS,ION CANNON BATTERIES,HEAVY TRACTORBEAM PROJECTORS,HULL-CUTTING AIRLOCKS,QUAD BLASTER CANNON,EMP CANNONS,MEDIUM BLASTER CANNONS,LIGHT TURBOLASER CANNONS,(VARIES ACCORDING TO SHIP MODEL),DOUBLE HEAVY LASER TURRETS,QUAD LASER CANNON TURRETS,HEAVY TURBOLASERS,MEDIUM DUAL TURBOLASERS,PROTON TORPEDO TUBES,DOUBLE LASER CANNON TURRETS,QUAD TURBOLASER TURRETS,TURBOLASER TURRETS,Ship Type
0,3.0,40.0,4.0,15000.0,0.222222,2.0,2.0,35.0,45.0,0.998658,-1.0,0.0,0.030741,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Landing Craft
1,18.0,7.8,1.0,150.0,0.166667,-1.0,-1.0,110.0,14.0,0.037238,24.0,0.0,0.030741,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TIE Fighters
2,8.0,298.0,150.0,100000000.0,6.000000,2.0,15.0,18.0,228.0,0.987765,320.0,300.0,0.030741,0.0,10.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Medium Ships
3,8.0,7.4,1.0,55.0,0.033333,-1.0,-1.0,115.0,16.0,0.037238,-1.0,0.0,0.030741,2.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,Other Starfighters
4,18.0,6.3,1.0,65.0,0.066667,-1.0,-1.0,75.0,9.0,0.037238,-1.0,0.0,0.030741,1.0,0.0,1.0,1.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,TIE Bombers


# Estadísticas de Datos Preparados

In [ ]:
#@title Mostrar estadísticas de datos preparados

generar_estadisticas_detalladas(ndf, "> Estadísticas de los datos preparados")



 > Estadísticas de los datos preparados : 


,Tipo Valor,Rango Valores,Promedio ± Desvío,¿Valores Ceros?,¿Valores Nulos?
Manufacturer,float64,[ 0.00 ; 29.00 ],14.626 ± 7.205,1,0
Length,float64,[ 2.10 ; 16000.00 ],481.423 ± 2123.802,0,0
Crew,float64,[ 0.00 ; 712645.00 ],14605.710 ± 90254.128,2,0
Cargo Capacity,float64,[ -1.00 ; 600000000.00 ],15213112.766 ± 75767701.080,0,0
Consumables,float64,[ 0.03 ; 120.00 ],8.328 ± 19.252,0,0
Hyperdrive Multiplier,float64,[ -1.00 ; 15.00 ],0.899 ± 2.004,0,0
Hyperdrive Backup,float64,[ -1.00 ; 20.00 ],4.972 ± 7.145,0,0
Speed,float64,[ 4.00 ; 155.00 ],69.159 ± 38.796,0,0
Hull,float64,[ 5.00 ; 77710.00 ],1473.692 ± 8888.793,0,0
Special Features,float64,[ 0.04 ; 1.00 ],0.225 ± 0.381,0,0


Tiene 75 atributos y 107 ejemplos.




#Exportar datos de Naves Imperiales

In [ ]:
#@title Exporta los datos como CSV

download_CSV = False #@param{type:"boolean"}

import os

archivo_datos_exportar = 'naves.csv'  #@param {type:"string"}

# crea directorio si no existe
if not os.path.isdir(path):
  os.mkdir(path)

def data2CSV(df, nomArch, descripcion):
  if df is None:
    print("No hay " + descripcion + " para exportar")
  df.to_csv(nomArch, index=False)
  if download_CSV:
    files.download(nomArch)
  print(descripcion + " exportados como ", nomArch)

# exporta datos
data2CSV(ndf, path + archivo_datos_exportar, "Datos preparados")
